In [1]:
# -*- coding: utf-8 -*-
"""
Spatial Cell Detection - Gradio Interface
Interactive web interface for detecting and visualizing spatial cell types
"""
# import nest_asyncio
# nest_asyncio.apply()
import gradio as gr
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import ndimage, signal
from scipy.ndimage import label
from skimage.measure import regionprops
import pickle
import io
from PIL import Image
import random
from typing import Tuple, Dict, List
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd())
import grid_cell_scorer as gcs

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def matlab_style_gauss2D(shape: Tuple[int, int], sigma: float) -> np.ndarray:
    """Create 2D Gaussian kernel matching MATLAB's fspecial('gaussian')."""
    m, n = [(ss - 1.) / 2. for ss in shape]
    y, x = np.ogrid[-m:m+1, -n:n+1]
    h = np.exp(-(x*x + y*y) / (2. * sigma * sigma))
    h[h < np.finfo(h.dtype).eps * h.max()] = 0
    sumh = h.sum()
    if sumh != 0:
        h /= sumh
    return h

def occupancy_map_func(pos: np.ndarray, reso: int = 40, win_len: int = 9) -> np.ndarray:
    """Calculate occupancy probability map from position data."""
    H1, _, _ = np.histogram2d(pos[:, 0], pos[:, 1], bins=reso)
    gaussian = matlab_style_gauss2D([win_len, win_len], 3.0)
    pos_prob = signal.convolve2d(gaussian, H1)
    pos_prob = ndimage.rotate(pos_prob, 0)
    pos_prob = pos_prob[int(win_len/2):-int(win_len/2), int(win_len/2):-int(win_len/2)]
    pos_prob = pos_prob / np.nansum(pos_prob)
    pos_prob[pos_prob == 0] = np.nan
    return pos_prob

def get_combined_layer_firing_rates(encoded):
    """Stack firing rates from all layers into single array."""
    combined = {}
    for key in ['D1', 'D2', 'D3', 'LEC', 'MEC', 'graph_LEC', 'graph_MEC']:
        combined[key] = encoded[key].T
    return np.vstack(list(combined.values()))

def global_index_to_label(encoded_data, global_index):
    """Map a 1-based global neuron index to (layer_name, within_layer_index)."""
    layer_order = ['D1', 'D2', 'D3', 'LEC', 'MEC', 'graph_LEC', 'graph_MEC']
    offset = 1
    for layer_name in layer_order:
        if layer_name not in encoded_data:
            continue
        n = encoded_data[layer_name].shape[1] if encoded_data[layer_name].shape[0] > encoded_data[layer_name].shape[1] else encoded_data[layer_name].shape[0]
        if offset <= global_index < offset + n:
            return layer_name, global_index - offset + 1
        offset += n
    return "unknown", global_index

# ============================================================================
# PLACE CELL FUNCTIONS
# ============================================================================

def firing_rate_map_place_cells(ot, all_dat, occupancy_map, thresh_param=0.0, res_param=40):
    """Generate firing rate map for place cell analysis."""
    res = res_param
    mean_resp = np.mean(ot)
    std_resp = np.std(ot)
    thresh = mean_resp + (thresh_param * std_resp)

    firr = np.nonzero(ot > thresh)
    firposgrid = all_dat[firr[0], :2]

    # Using real coordinate bounds
    xmin, xmax = all_dat[:, 0].min(), all_dat[:, 0].max()
    ymin, ymax = all_dat[:, 1].min(), all_dat[:, 1].max()

    # Creating spatial grid in real space
    x = np.linspace(xmin, xmax, res)
    y = np.linspace(ymin, ymax, res)

    # x = np.arange(-1, 1, 1/res)
    # y = np.arange(-1, 1, 1/res)
    fx, fy = np.meshgrid(x, y)
    firingmap = np.zeros(fx.shape)
    win_len = 9
    firingvalue = ot[firr]

    for ii in range(len(firposgrid)):
        q1 = np.argmin(abs(firposgrid[ii, 0] - fx[1, :]))
        q2 = np.argmin(abs(firposgrid[ii, 1] - fy[:, 1]))
        firingmap[q1, q2] = max(firingvalue[ii], firingmap[q1, q2])

    firingmap = firingmap / occupancy_map
    gaussian = matlab_style_gauss2D([9, 9], 3)
    spikes_smooth = signal.convolve2d(gaussian, firingmap)
    rotated_img = ndimage.rotate(spikes_smooth, 90)
    rotated_img = rotated_img[int(win_len/2):-int(win_len/2), int(win_len/2):-int(win_len/2)]
    extent = [xmin, xmax, ymin, ymax]
    return rotated_img, extent

def sparsity_func(pos_prob, firing_rate):
    """Calculate sparsity metric for spatial firing."""
    pos_prob2 = np.copy(pos_prob)
    pos_prob2 = pos_prob2 / np.nansum(pos_prob2)
    avg_rate = np.nansum(np.ravel(firing_rate * pos_prob2))
    numerator = avg_rate ** 2
    denominator = np.nansum(np.multiply(pos_prob2, firing_rate**2))
    return numerator / denominator

def _inf_rate(rate_map, px):
    """Calculate spatial information rate."""
    tmp_map = np.array(rate_map, copy=True, dtype=float)
    tmp_map[np.isnan(tmp_map)] = 0.0
    avg_rate = np.sum(tmp_map * px.ravel() if px.shape != tmp_map.shape else tmp_map * px)

    # Only compute log where firing rate is positive — 0*log(0) = 0 by convention
    valid = tmp_map > 0
    info = np.zeros_like(tmp_map)
    info[valid] = tmp_map[valid] * np.log2(tmp_map[valid] / avg_rate) * (px.ravel() if px.shape != tmp_map.shape else px)[valid]

    return float(np.sum(info)), float(avg_rate)

def get_place_cells_quick(pos_out, encoded_data, lim=1.5, reso=40, si_thresh=0.3, sp_thresh=0.1):
    """Quick place cell detection without shuffling validation."""
    lay_nam = ['D1', 'D2', 'D3', 'LEC', 'MEC', 'graph_LEC', 'graph_MEC']

    spatial_info    = []
    sparsity_scores = []
    hgs_scores      = []
    occupancy_map   = occupancy_map_func(pos_out, reso=reso)
    rng             = np.random.default_rng(0)

    for k in range(len(lay_nam)):
        layer_arr = encoded_data[lay_nam[k]]  # shape: (timesteps, neurons)

        for i in range(layer_arr.shape[1]):
            img_dat, _ = firing_rate_map_place_cells(
                layer_arr[:, i], pos_out, occupancy_map=occupancy_map,
                thresh_param=lim, res_param=reso
            )

            sparsity_cell = sparsity_func(occupancy_map, img_dat)
            if not np.isfinite(sparsity_cell):
                sparsity_cell = 0.0
            sparsity_scores.append(round(float(sparsity_cell), 2))

            sp_info_rate, avg_rate = _inf_rate(img_dat, occupancy_map)
            spatial_score = (sp_info_rate * 0.1/ avg_rate) if avg_rate > 0 else 0.0
            if not np.isfinite(spatial_score):
                spatial_score = 0.0
            spatial_info.append(round(float(spatial_score), 2))

            # HGS for place cells — score without shuffle test, just the raw value
            try:
                _, corr_ring, _ = gcs.build_autocorrelation_ring(img_dat, win_len=7)
                hgs = float(gcs.gridscore(corr_ring))
            except Exception:
                hgs = 0.0
            hgs_scores.append(round(hgs, 4) if np.isfinite(hgs) else 0.0)

    selected_neus_spars   = np.where(np.asarray(sparsity_scores) < sp_thresh)[0]
    selected_neus_spainfo = np.where(np.asarray(spatial_info)    > si_thresh)[0]
    place_cells = list(set(selected_neus_spars) & set(selected_neus_spainfo))
    place_cells = [x + 1 for x in place_cells]

    return place_cells, spatial_info, sparsity_scores, hgs_scores

# ============================================================================
# GRID CELL FUNCTIONS  (delegated to grid_cell_scorer.py)
# ============================================================================

def firing_rate_map_grid(ot, all_dat, occupancy_map, thresh_param=1.5, res_param=40):
    """Generate firing rate map for grid cell analysis using the canonical scorer."""
    map_res = int(occupancy_map.shape[0]) if occupancy_map is not None else res_param
    spikes_smooth, extent = gcs.build_firing_rate_map(
        responses=ot,
        pos=all_dat,
        occupancy_map=occupancy_map,
        thresh_param=thresh_param,
        res_param=map_res,
    )
    return spikes_smooth, extent


def detect_grid_cells(pos_out, encoded_data, lim=1.5, reso=40, num_shuffles=20, random_seed=0):
    """
    Detect grid cells using HGS with shuffle-based significance test.
    Returns:
        grid_cells       — list of 1-based global neuron indices
        autocorr_data    — dict mapping global_index -> {autocorr_map, autocorr_ring, hgs, threshold_95, layer, neuron_within_layer}
    """
    layer_order = ('D1', 'D2', 'D3', 'LEC', 'MEC', 'graph_LEC', 'graph_MEC')
    occupancy_map = gcs.occupancy_map_func(pos_out, reso=reso)
    rng = np.random.default_rng(random_seed)

    grid_cells = []
    autocorr_data = {}
    global_index = 1

    for layer_name in layer_order:
        if layer_name not in encoded_data:
            continue
        layer_arr = np.asarray(encoded_data[layer_name], dtype=float)
        n_neurons = layer_arr.shape[1]  # shape is (timesteps, neurons)

        for neuron_idx in range(n_neurons):
            result = gcs.score_single_neuron(
                responses=layer_arr[:, neuron_idx],
                pos=pos_out,
                occupancy_map=occupancy_map,
                num_shuffles=num_shuffles,
                rng=rng,
                thresh_param=lim,
                res_param=reso,
            )
            if result['is_grid_cell']:
                grid_cells.append(global_index)
                autocorr_data[global_index] = {
                    'autocorr_map':  result['autocorr_map'],
                    'autocorr_ring': result['autocorr_ring'],
                    'hgs':           result['original_hgs'],
                    'threshold_95':  result['shuffle_threshold_95'],
                    'layer':         layer_name,
                    'neuron_within_layer': neuron_idx + 1,
                }
            global_index += 1

    return grid_cells, autocorr_data

# ============================================================================
# BORDER CELL FUNCTIONS
# ============================================================================

def border_score_func(img):
    """Calculate border score based on wall coverage."""
    thresholded_img = (img > 0.1 * np.amax(img)).astype(int)
    labelled_img, labels = label(thresholded_img)

    temp_score = []

    for i in range(1, labels + 1):
        temp_label = (labelled_img == i).astype(int)

        if np.sum(temp_label) > 30:
            coverage_wall_1 = np.sum(temp_label[:, :1])  / temp_label.shape[0]
            coverage_wall_2 = np.sum(temp_label[:1, :])  / temp_label.shape[1]
            coverage_wall_3 = np.sum(temp_label[:, -1:]) / temp_label.shape[0]
            coverage_wall_4 = np.sum(temp_label[-1:, :]) / temp_label.shape[1]

            coverage_walls = np.asarray([coverage_wall_1, coverage_wall_2,
                                         coverage_wall_3, coverage_wall_4])

            norm_img   = img / np.linalg.norm(img)
            properties = regionprops(temp_label, norm_img)
            center_of_mass = np.asarray(list(properties[0].centroid_weighted))  # fixed: removed + 0.5

            dist_from_wall = [
                center_of_mass[1],
                center_of_mass[0],
                temp_label.shape[1] - center_of_mass[1],
                temp_label.shape[0] - center_of_mass[0]
            ]

            dm = np.divide(
                np.asarray(dist_from_wall),
                np.asarray([temp_label.shape[1], temp_label.shape[0],
                            temp_label.shape[1], temp_label.shape[0]])
            )

            num = coverage_walls - dm
            den = coverage_walls + dm
            temp_score.append(num / den)

    return np.asarray(temp_score)

def get_border_cells(pos, encoded_data, lim=1.5, reso=40, cs_thresh=0.5, si_thresh=0.3):
    """Detect border cells."""
    lay_nam       = ['D1', 'D2', 'D3', 'LEC', 'MEC', 'graph_LEC', 'graph_MEC']
    border_cells  = []
    border_scores = []   # max border score per detected cell
    occupancy_map = occupancy_map_func(pos, reso=reso)
    cell_idx      = 1

    for k in range(len(lay_nam)):
        layer_arr = encoded_data[lay_nam[k]]

        for i in range(layer_arr.shape[1]):
            firing_map, _ = firing_rate_map_place_cells(
                layer_arr[:, i], pos, occupancy_map, lim, reso
            )
            temp_score    = border_score_func(firing_map)
            sp_info_rate, avg_rate = _inf_rate(firing_map, occupancy_map)
            spatial_info  = ((sp_info_rate * 0.1)/ avg_rate) if avg_rate > 0 else 0.0

            if len(temp_score) and np.max(temp_score) >= cs_thresh and spatial_info < si_thresh:
                border_cells.append(cell_idx)
                border_scores.append(float(np.max(temp_score)))

            cell_idx += 1

    return border_cells, border_scores

# ============================================================================
# OBJECT CELL FUNCTIONS
# ============================================================================

def firing_rate_map_object(ot, all_dat, occupancy_map=None, thresh_param=0.0, res_param=40):
    """Generate firing rate map for object cell analysis."""
    res = res_param

    mean_resp = np.mean(ot)
    std_resp = np.std(ot)
    thresh = mean_resp + (thresh_param * std_resp)

    firr = np.where(ot > thresh)[0]
    firposgrid = all_dat[firr, :]   # positions where firing > thresh
    firingvalue = ot[firr]

    xmin, xmax = all_dat[:, 0].min(), all_dat[:, 0].max()
    ymin, ymax = all_dat[:, 1].min(), all_dat[:, 1].max()

    x = np.linspace(xmin, xmax, res)
    y = np.linspace(ymin, ymax, res)

    fx, fy = np.meshgrid(x, y)   # fx: x coords (cols), fy: y coords (rows)
    firingmap = np.zeros((res, res))

    # fill map (row = y index, col = x index)
    for ii in range(len(firposgrid)):
        col = np.argmin(np.abs(firposgrid[ii, 0] - fx[0, :]))   # x axis
        row = np.argmin(np.abs(firposgrid[ii, 1] - fy[:, 0]))   # y axis
        firingmap[row, col] = max(firingvalue[ii], firingmap[row, col])

    # normalize
    maxval = np.max(firingmap)
    if maxval > 0:
        firingmap = firingmap / maxval

    gaussian = matlab_style_gauss2D([5, 5], 1.5)
    spikes_smooth = signal.convolve2d(firingmap, gaussian, mode="same")

    extent = [xmin, xmax, ymin, ymax]
    return spikes_smooth, extent


def get_object_bounds(obj_center, num_bins, extent, radius=0.2):
    """
    Convert object center in real coordinates -> bin index bounds.
    extent = [xmin, xmax, ymin, ymax]
    """
    xmin, xmax, ymin, ymax = extent

    # map real coord -> bin index
    def to_bin_x(x):
        return (x - xmin) / (xmax - xmin) * (num_bins - 1)

    def to_bin_y(y):
        return (y - ymin) / (ymax - ymin) * (num_bins - 1)

    x_min = to_bin_x(obj_center[0] - radius)
    x_max = to_bin_x(obj_center[0] + radius)
    y_min = to_bin_y(obj_center[1] - radius)
    y_max = to_bin_y(obj_center[1] + radius)

    # clamp
    x_min = max(0, x_min)
    y_min = max(0, y_min)
    x_max = min(num_bins - 1, x_max)
    y_max = min(num_bins - 1, y_max)

    return x_min, x_max, y_min, y_max


def z_score_object(rate_map, obj_c, extent, radius=0.2):
    """Calculate z-score for object region."""
    nbins = rate_map.shape[0]

    x_min, x_max, y_min, y_max = get_object_bounds(obj_c, nbins, extent, radius)

    obj_vals = []
    out_vals = []

    for row in range(rate_map.shape[0]):
        for col in range(rate_map.shape[1]):
            if (x_min <= col <= x_max) and (y_min <= row <= y_max):
                obj_vals.append(rate_map[row, col])
            else:
                out_vals.append(rate_map[row, col])

    obj_vals = np.array(obj_vals, dtype=float)
    out_vals = np.array(out_vals, dtype=float)

    obj_vals = obj_vals[~np.isnan(obj_vals)]
    out_vals = out_vals[~np.isnan(out_vals)]

    if len(obj_vals) == 0 or len(out_vals) == 0:
        return np.nan

    # if outside region smaller, allow replacement
    replace_flag = len(out_vals) < len(obj_vals)

    rand_out = np.random.choice(out_vals, size=len(obj_vals), replace=replace_flag)

    out_mean = np.mean(rand_out)
    out_std = np.std(rand_out)
    obj_mean = np.mean(obj_vals)

    den = out_std if out_std > 0.02 else (out_std + 0.02)

    z_score = (obj_mean - out_mean) * np.sqrt(len(obj_vals)) / den
    return z_score


def get_object_cells(pos, encoded_data, lim=1.5, reso=40, z_score_thresh=10, obj_center=None, radius=0.2):
    """Detect object cells based on z-scores."""
    if obj_center is None:
        raise ValueError("obj_center must be provided like [x,y]")

    lay_nam = ['D1', 'D2', 'D3', 'LEC', 'MEC', 'graph_LEC', 'graph_MEC']
    z_scores = []

    for k in range(len(lay_nam)):
        resp_neurons = encoded_data[lay_nam[k]].T

        for i in range(resp_neurons.shape[0]):
            img_dat, extent = firing_rate_map_object(
                resp_neurons[i], pos, None, lim, reso
            )

            img_dat = np.nan_to_num(img_dat, nan=0.0, posinf=0.0, neginf=0.0)

            max_val = np.max(img_dat)
            if max_val > 0:
                img_dat = img_dat / max_val

            z = z_score_object(img_dat, obj_center, extent, radius=radius)
            z_scores.append(round(z, 2))

    object_cells = [i + 1 for i, z in enumerate(z_scores) if z >= z_score_thresh]
    return object_cells, z_scores

# ============================================================================
# VISUALIZATION FUNCTIONS
# ============================================================================

def create_firing_rate_visualization(pos, encoded_data, cell_indices, cell_type, lim, reso, page_size=32):
    """Create firing rate map visualizations, split into pages to avoid WebP size limits."""
    if not cell_indices:
        return []

    occupancy_map  = occupancy_map_func(pos, reso=reso)
    combined_rates = get_combined_layer_firing_rates(encoded_data)
    images         = []

    for page_start in range(0, len(cell_indices), page_size):
        page_cells = cell_indices[page_start:page_start + page_size]
        n_cells    = len(page_cells)
        cols       = min(4, n_cells)
        rows       = (n_cells + cols - 1) // cols

        fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
        if rows == 1 and cols == 1:
            axes = np.array([[axes]])
        elif rows == 1:
            axes = axes.reshape(1, -1)
        elif cols == 1:
            axes = axes.reshape(-1, 1)

        for idx, cell_idx in enumerate(page_cells):
            row = idx // cols
            col = idx % cols
            ax  = axes[row, col]

            neuron_idx  = cell_idx - 1
            layer_name, neuron_within_layer = global_index_to_label(encoded_data, cell_idx)

            if cell_type == "Object Cell":
                firing_map, extent = firing_rate_map_object(
                    combined_rates[neuron_idx, :], pos, occupancy_map, lim, reso)
            elif cell_type == "Place Cell":
                firing_map, extent = firing_rate_map_place_cells(
                    combined_rates[neuron_idx, :], pos, occupancy_map, lim, reso)
            elif cell_type == "Grid Cell":
                firing_map, extent = firing_rate_map_grid(
                    combined_rates[neuron_idx, :], pos, occupancy_map, lim, reso)
            else:
                firing_map, extent = firing_rate_map_place_cells(
                    combined_rates[neuron_idx, :], pos, occupancy_map, lim, reso)

            im = ax.imshow(firing_map, cmap='jet', origin='lower', extent=extent)
            ax.set_xlim(extent[0], extent[1])
            ax.set_ylim(extent[2], extent[3])
            ax.set_aspect('equal')
            ax.set_title(f'{layer_name} #{neuron_within_layer}', fontsize=8)
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        for idx in range(n_cells, rows * cols):
            axes[idx // cols, idx % cols].axis('off')

        plt.tight_layout()
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        buf.seek(0)
        plt.close()
        images.append(Image.open(buf).copy())

    return images

def create_autocorr_visualization(autocorr_data, page_size=16):
    """Plot autocorrelograms, split into pages to avoid WebP size limits."""
    if not autocorr_data:
        return []

    items  = list(autocorr_data.items())
    images = []

    for page_start in range(0, len(items), page_size):
        page_items = items[page_start:page_start + page_size]
        n_cells    = len(page_items)
        cols       = min(4, n_cells)
        rows       = (n_cells + cols - 1) // cols

        fig, axes = plt.subplots(rows, cols * 2, figsize=(4 * cols * 2, 4 * rows))
        if rows == 1 and cols * 2 == 1:
            axes = np.array([[axes]])
        elif rows == 1:
            axes = axes.reshape(1, -1)
        elif cols * 2 == 1:
            axes = axes.reshape(-1, 1)

        for idx, (global_idx, data) in enumerate(page_items):
            row       = idx // cols
            left_col  = (idx % cols) * 2
            right_col = left_col + 1
            label     = f"{data['layer']} #{data['neuron_within_layer']}\nHGS={data['hgs']:.3f}  thr={data['threshold_95']:.3f}"

            ax_l = axes[row, left_col]
            ax_l.imshow(data['autocorr_map'], cmap='coolwarm', origin='lower', vmin=-1, vmax=1)
            ax_l.set_title(f'{label}\n(autocorr)', fontsize=8)
            ax_l.axis('off')

            ax_r = axes[row, right_col]
            ax_r.imshow(data['autocorr_ring'], cmap='coolwarm', origin='lower', vmin=-1, vmax=1)
            ax_r.set_title(f'{label}\n(ring)', fontsize=8)
            ax_r.axis('off')

        for idx in range(n_cells, rows * cols):
            axes[idx // cols, (idx % cols) * 2    ].axis('off')
            axes[idx // cols, (idx % cols) * 2 + 1].axis('off')

        plt.suptitle('Grid Cell Autocorrelograms', fontsize=12, fontweight='bold', y=1.01)
        plt.tight_layout()
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        buf.seek(0)
        plt.close()
        images.append(Image.open(buf).copy())

    return images

def create_layer_distribution(cell_indices, units, units2):
    """Create distribution plot of cells across layers."""
    layers = {
        'D1': (0, units2),
        'D2': (units2, int(units2*2)),
        'D3': (int(units2*2), int(units2*3)),
        'LEC': (int(units2*3), int((units2*3) + units)),
        'MEC': (int((units2*3) + (units)), int((units2*3) + (units*2))),
        'GR_LEC': (int((units2*3) + (units*2)), int((units2*3) + (units*3))),
        'GR_MEC': (int((units2*3) + (units*3)), int((units2*3) + (units*4)))
    }

    counts = {}
    for layer, (start, end) in layers.items():
        counts[layer] = sum(1 for idx in cell_indices if start <= idx < end)

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(counts.keys(), counts.values(), color='steelblue', edgecolor='black')

    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}', ha='center', va='bottom', fontsize=10)

    ax.set_xlabel('Layer', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Cells', fontsize=12, fontweight='bold')
    ax.set_title('Distribution of Detected Cells Across Layers', fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    plt.close()

    return Image.open(buf)

def create_metrics_summary(
    place_cells=None,  spatial_info=None,  place_hgs=None,
    grid_cells=None,   autocorr_data=None,
    border_cells=None, border_scores=None,
    object_cells=None,
):
    """Render a metrics summary table as a matplotlib figure."""
    rows = []

    if place_cells is not None and spatial_info is not None:
        detected_si  = [spatial_info[i - 1] for i in place_cells if i - 1 < len(spatial_info)]
        mean_si      = float(np.mean(detected_si)) if detected_si else 0.0
        detected_hgs = [place_hgs[i - 1] for i in place_cells if place_hgs and i - 1 < len(place_hgs)]
        mean_ph      = float(np.mean(detected_hgs)) if detected_hgs else 0.0
        rows.append(("Place Cells",  "Mean Spatial Info\n(bits/spike)", f"{mean_si:.4f}", len(place_cells)))
        rows.append(("Place Cells",  "Mean HGS",                        f"{mean_ph:.4f}", len(place_cells)))

    if grid_cells is not None and autocorr_data is not None:
        hgs_vals = [v['hgs'] for v in autocorr_data.values() if np.isfinite(v['hgs'])]
        mean_hgs = float(np.mean(hgs_vals)) if hgs_vals else 0.0
        rows.append(("Grid Cells",   "Mean HGS",                        f"{mean_hgs:.4f}", len(grid_cells)))

    if border_cells is not None and border_scores is not None:
        mean_bs = float(np.mean(border_scores)) if border_scores else 0.0
        rows.append(("Border Cells", "Mean Border Score",                f"{mean_bs:.4f}", len(border_cells)))

    if object_cells is not None:
        rows.append(("Object Cells", "—",                                "—",              len(object_cells)))

    if not rows:
        return None

    n_rows = len(rows)
    fig_h  = max(2.5, 0.7 * n_rows + 1.5)
    fig, ax = plt.subplots(figsize=(10, fig_h))
    ax.axis('off')

    col_labels  = ["Cell Type", "Metric", "Value", "Count"]
    col_widths  = [0.22, 0.40, 0.20, 0.18]   # proportional widths

    table = ax.table(
        cellText=rows,
        colLabels=col_labels,
        colWidths=col_widths,
        cellLoc='center',
        loc='center',
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 2.2)   # taller rows so wrapped text isn't clipped

    # header styling
    for col in range(len(col_labels)):
        cell = table[0, col]
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')

    # alternating row colours
    for row_idx in range(1, n_rows + 1):
        colour = '#ecf0f1' if row_idx % 2 == 0 else '#ffffff'
        for col in range(len(col_labels)):
            table[row_idx, col].set_facecolor(colour)

    ax.set_title('Detection Metrics Summary', fontsize=13, fontweight='bold', pad=16)
    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=130, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return Image.open(buf)

# ============================================================================
# MAIN PROCESSING FUNCTION
# ============================================================================

def process_spatial_cells(env_type, pkl_file, cell_types, firing_rate_thresh, res_param, information_rate_thresh_pc, sparsity_thresh, num_cont_ff,
                    z_score_thresh, information_rate_thresh_bc, coverage_score_thresh, units, units2, obj_center):
    """Main processing function for Gradio interface."""
    # Environment mapping
    # Environment mapping
    env_configs = {
        "Circular Env (2 Objects at [0.5, 0.5], [-0.6, -0.3])": {
            "traj_file": "traj_circleenv_sqrt1.5_30ksteps_2SquObj(0.3_size)_loc_(0.5,0.5)_(-0.6,-0.3).pkl",
            "frames_file": "frames_circleenv_sqrt1.5_30ksteps_2SquObj(0.3_size)_loc_(0.5,0.5)_(-0.6,-0.3).pkl",
            "env_id": "circleenv_2sqobj_p5p5_np6np3_30k",
            "obj_center": [(0.5, 0.5), (-0.6, -0.3)],
            "obj_pres": True
        },
        "Square Env (2 Objects at [-0.5, 0.8], [0.8, 0.0])": {
            "traj_file": "traj_squareenv_2x2_30ksteps_2SquObj(0.2_size)_loc_(-0.5,0.8)_(0,0.8).pkl",
            "frames_file": "frames_squareenv_2x2_30ksteps_2SquObj(0.2_size)_loc_(-0.5,0.8)_(0,0.8).pkl",
            "env_id": "squareenv_circleenv_2sqobj_np5p8_p80_30k",
            "obj_center": [(-0.5, 0.8), (0.8, 0.0)],
            "obj_pres": True
        }
    }

    obj_configs = {

        "[0.5, 0.5]": {
            "obj_center": (0.5, 0.5)
        },

        "[-0.6, -0.3]": {
            "obj_center": (-0.6, -0.3)
        },

        "[0.8, 0.0]": {
            "obj_center": (0.8, 0.0)
        },
        "[-0.5, 0.8]": {
            "obj_center": (-0.5, 0.8)
        }
    }
    env_config = env_configs[env_type]
    object_center = obj_configs[obj_center]

    try:
        # Load data
        with open(pkl_file, 'rb') as f:
            encoded_data = pickle.load(f)

        with open(env_config['traj_file'], 'rb') as f:
            pos_data = pickle.load(f)

        # Extract position data
        if isinstance(pos_data, dict):
            if 'x' in pos_data and 'y' in pos_data:
                x = np.asarray(pos_data['x'])
                y = np.asarray(pos_data['y'])
                pos = np.column_stack((x, y))
            else:
                pos = pos_data['pos']
        else:
            pos = pos_data

        results = {}
        all_visualizations = []

        # initialise metric accumulators
        place_cells_out  = None;  spatial_info_out  = None; place_hgs_out = None;
        grid_cells_out   = None;  autocorr_data_out = None
        border_cells_out = None;  border_scores_out = None
        object_cells_out = None

        # Process each selected cell type
        if "Place Cell" in cell_types:
            place_cells, spatial_info, sparsity, hgs_scores = get_place_cells_quick(
                pos, encoded_data, firing_rate_thresh, res_param, information_rate_thresh_pc, sparsity_thresh
            )
            place_cells_out, spatial_info_out, place_hgs_out, = place_cells, spatial_info, hgs_scores
            results['Place Cells'] = len(place_cells)

            dist_img = create_layer_distribution(place_cells, units, units2)
            fire_img = create_firing_rate_visualization(
                pos, encoded_data, place_cells, "Place Cell", firing_rate_thresh, res_param
            )

            all_visualizations.append((dist_img, "Place Cells Distribution"))

            for i, img in enumerate(fire_img):
                label = "Place Cells Firing Maps" if i == 0 else f"Place Cells Firing Maps (page {i+1})"
                all_visualizations.append((img, label))

        if "Grid Cell" in cell_types:
            grid_cells, autocorr_data = detect_grid_cells(pos, encoded_data, lim=firing_rate_thresh, reso=res_param, num_shuffles=num_cont_ff)
            grid_cells_out, autocorr_data_out = grid_cells, autocorr_data
            results['Grid Cells'] = len(grid_cells)

            dist_img = create_layer_distribution(grid_cells, units, units2)
            fire_img = create_firing_rate_visualization(
                pos, encoded_data, grid_cells, "Grid Cell", firing_rate_thresh, res_param
            )

            autocorr_img = create_autocorr_visualization(autocorr_data)

            all_visualizations.append((dist_img, "Grid Cells Distribution"))
            for i, img in enumerate(fire_img):
                label = "Grid Cells Firing Maps" if i == 0 else f"Grid Cells Firing Maps (page {i+1})"
                all_visualizations.append((img, label))
            for i, img in enumerate(autocorr_img):
                label = "Grid Cells Autocorrelograms" if i == 0 else f"Grid Cells Autocorrelograms (page {i+1})"
                all_visualizations.append((img, label))

        if "Border Cell" in cell_types:
            border_cells, border_scores = get_border_cells(pos, encoded_data, firing_rate_thresh, res_param, coverage_score_thresh, information_rate_thresh_bc)
            border_cells_out, border_scores_out = border_cells, border_scores
            results['Border Cells'] = len(border_cells)

            dist_img = create_layer_distribution(border_cells, units, units2)
            fire_img = create_firing_rate_visualization(
                pos, encoded_data, border_cells, "Border Cell", firing_rate_thresh, res_param
            )

            all_visualizations.append((dist_img, "Border Cells Distribution"))
            for i, img in enumerate(fire_img):
                label = "Border Cells Firing Maps" if i == 0 else f"Border Cells Firing Maps (page {i+1})"
                all_visualizations.append((img, label))

        if "Object Cell" in cell_types:
            object_cells, z_scores = get_object_cells(pos, encoded_data, firing_rate_thresh, res_param, z_score_thresh, object_center['obj_center'])
            object_cells_out = object_cells
            results['Object Cells'] = len(object_cells)

            dist_img = create_layer_distribution(object_cells, units, units2)
            fire_img = create_firing_rate_visualization(
                pos, encoded_data, object_cells, "Object Cell", firing_rate_thresh, res_param
            )

            all_visualizations.append((dist_img, "Object Cells Distribution"))
            for i, img in enumerate(fire_img):
                label = "Object Cells Firing Maps" if i == 0 else f"Object Cells Firing Maps (page {i+1})"
                all_visualizations.append((img, label))

        metrics_img = create_metrics_summary(
            place_cells=place_cells_out,   spatial_info=spatial_info_out, place_hgs=place_hgs_out,
            grid_cells=grid_cells_out,     autocorr_data=autocorr_data_out,
            border_cells=border_cells_out, border_scores=border_scores_out,
            object_cells=object_cells_out,
            )
        if metrics_img:
            all_visualizations.insert(0, (metrics_img, "Metrics Summary"))
        # Create summary text
        summary = "## Detection Results\n\n"
        for cell_type, count in results.items():
            summary += f"**{cell_type}**: {count} neurons detected\n\n"

        return summary, all_visualizations

    except Exception as e:
        return f"Error processing files: {str(e)}", []

# ============================================================================
# GRADIO INTERFACE
# ============================================================================

def create_interface():
    """Creating Gradio interface."""

    with gr.Blocks(title="Spatial Cell Detection in the Deep EC-HPC Network Model") as demo:
        gr.Markdown(
            """
            # 🧠 Spatial Cell Detection & Visualization

            Upload neural activation data for desired environment to detect and visualize different types of spatial cells.

            **Supported Cell Types:**
            - **Place Cells**: Fire at specific locations
            - **Grid Cells**: Fire in periodic spatial patterns
            - **Border Cells**: Fire along environment boundaries
            - **Object Cells**: Fire near objects
            """
        )

        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### 📁 Data Input")

                # Environment selection
                env_type = gr.Dropdown(
                    choices=["Circular Env (2 Objects at [0.5, 0.5], [-0.6, -0.3])", "Square Env (2 Objects at [-0.5, 0.8], [0.8, 0.0])"],
                    label="Environment Type",
                    value="Circular Env (2 Objects at [0.5, 0.5], [-0.6, -0.3])",
                    info="Select the environment for cell detection"
                )

                encoded_file = gr.File(
                    label="Neural Activations (PKL file)",
                    file_types=[".pkl", ".pickle"],
                    type="filepath"
                )

                # Model architecture
                gr.Markdown("### Model Architecture")
                with gr.Row():
                    units = gr.Slider(8, 512, value=32, step=16, label="LEC/MEC Units")
                    units2 = gr.Slider(8, 512, value=64, step=32, label="Dense Layer Units")

                gr.Markdown("### Detection Parameters")

                cell_types = gr.CheckboxGroup(
                    choices=["Place Cell", "Grid Cell", "Border Cell", "Object Cell"],
                    value=["Place Cell"],
                    label="Select Cell Types to Detect"
                )

                firing_rate_thresh = gr.Slider(
                    minimum=0.0,
                    maximum=5.0,
                    value=1.5,
                    step=0.1,
                    label="Firing Rate Map Threshold Parameter (std deviations above mean)"
                )

                res_param = gr.Slider(
                    minimum=20,
                    maximum=120,
                    value=40,
                    step=5,
                    label="Resolution Parameter (spatial binning)"
                )

                information_rate_thresh_pc = gr.Slider(
                    minimum=0.001,
                    maximum=1,
                    value=0.5,
                    step=0.001,
                    label="Spatial Information Rate Threshold (Place Cell)"
                )

                sparsity_thresh = gr.Slider(
                    minimum=0.001,
                    maximum=1,
                    value=0.5,
                    step=0.001,
                    label="Sparsity Threshold (Place Cell)"
                )

                num_cont_ff = gr.Slider(
                    minimum=2,
                    maximum=200,
                    value=20,
                    step=1,
                    label="Number of Shuffles for HGS significance test (Grid Cell)"
                )

                information_rate_thresh_bc = gr.Slider(
                    minimum=0.001,
                    maximum=1,
                    value=0.5,
                    step=0.001,
                    label="Spatial Information Rate Threshold (Border Cell)"
                )

                coverage_score_thresh = gr.Slider(
                    minimum=0.001,
                    maximum=1,
                    value=0.5,
                    step=0.001,
                    label="Coverage Score Threshold (Border Cell)"
                )

                z_score_thresh = gr.Slider(
                    minimum=0,
                    maximum=50,
                    value=25,
                    step=1,
                    label="Z-score Threshold (Object Cell)"
                )

                obj_center = gr.Dropdown(
                    choices=["[0.5, 0.5]", "[-0.5, 0.8]", "[0.8, 0.0]", "[-0.6, -0.3]"],
                    label="Object",
                    value="[0.5, 0.5]",
                    info="Select the object to detect Object Cells"
                )

                process_btn = gr.Button("🔍 Detect Cells", variant="primary", size="lg")

            with gr.Column(scale=2):
                gr.Markdown("### 📊 Results")

                summary_output = gr.Markdown(label="Summary")

                gallery_output = gr.Gallery(
                    label="Visualizations",
                    columns=1,
                    height="auto",
                    object_fit="contain"
                )

        gr.Markdown(
            """
            ---
            ## 📖 **TASK GUIDE**

            1. **Upload Files**: Select the environment and load the neural activations as a pkl file
            2. **Select Cell Types**: Choose which spatial cell types you want to detect
            3. **Adjust Resolution & Threshold Hyperparameters**
            4. **Detect**: Click the button to run analysis
            5. **View Results**: See Counts, Distributions, and Firing Rate Maps. Save and Note the results in your report.
            6. **Report**: Add the hyperparameters, counts, and visualizations to your report.


            ### 📦 Expected Data Format:

            **Neural Activations (encoded_data.pkl)**: Dictionary with keys:
            - 'D1', 'D2', 'D3', 'LEC', 'MEC', 'graph_LEC', 'graph_MEC'
            - Each value: numpy array of shape (n_timepoints, n_neurons)

            **Position Data (pos_data.pkl)**: Dictionary with keys 'x' and 'y', or direct (N, 2) array
            """
        )

        process_btn.click(
            fn=process_spatial_cells,
            inputs=[env_type, encoded_file, cell_types, firing_rate_thresh, res_param, information_rate_thresh_pc, sparsity_thresh, num_cont_ff,
                    z_score_thresh, information_rate_thresh_bc, coverage_score_thresh, units, units2, obj_center],
            outputs=[summary_output, gallery_output]
        )

    return demo

# ============================================================================
# LAUNCH
# ============================================================================

if __name__ == "__main__":
    demo = create_interface()
    demo.launch(
        share=True,
        debug=True, theme=gr.themes.Glass()
    )


ModuleNotFoundError: No module named 'gradio'